# SiGMA-lite on TU datasets (Tables 17 / 18) — Google Colab

Reproduces the TU protocol of Tables 17 / 18 of the SiGMA paper for **SiGMA-lite**: ungated MMA heads (2 attention + GCN, GIN, SAGE, GAT) with a single Gated-GPS-style gate on the mixed block update,
$x \leftarrow x + W_1\big(\mathrm{out\_proj}(\mathrm{concat\ heads}) \odot \sigma(W_g \,\mathrm{Norm}(x))\big)$.

Protocol: L=12, H=64, random 50/25/25 split, up to 1000 epochs, `reduce_on_plateau`, lr ∈ {1e-3, 1e-2}, seeds 0–4. Table 17 uses `d_h=16`, Table 18 `d_h=4`. The reported number is the test accuracy at the best validation epoch; the best learning rate (by mean) is reported per dataset.

**Runtime → Change runtime type → GPU** before running.

In [ ]:
# 1. Clone (or update) the repo
import os
import subprocess
import sys

REPO_URL = "https://github.com/RaphaelPellegrin/GNNPlus.git"
BRANCH = "paper_sigma_lite"
REPO_DIR = "/content/GNNPlus"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Install (keeps Colab's preinstalled CUDA torch; torch_scatter is optional)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[colab]"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)  # no runtime restart needed after the editable install

import torch
import torch_geometric

print("torch", torch.__version__, "| PyG", torch_geometric.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# 3. Where results and datasets go. Colab's disk is wiped on disconnect: set USE_DRIVE = True
#    to keep results on Google Drive (finished runs are skipped when you re-run the sweep).
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_ROOT = "/content/drive/MyDrive/sigma_lite_tu/results"
else:
    OUT_ROOT = f"{REPO_DIR}/results/sigma_lite_tu"
DATASET_DIR = f"{REPO_DIR}/datasets"
print("results ->", OUT_ROOT)

In [ ]:
# 4. Optional: log to Weights & Biases (project GNNPlus). Gate statistics are logged per layer
#    as gates/layer<i>/lite_gate_mean.
USE_WANDB = False
WANDB_ENTITY = "weber-geoml-harvard-university"  # or None for your default entity

if USE_WANDB:
    import wandb

    wandb.login()

In [ ]:
# 5. Smoke test: 5 epochs on MUTAG
from GNNPlus.sigma_lite_tu import TU_DATASETS, RunSpec, run_one, run_sweep, summarize

smoke = run_one(
    RunSpec("MUTAG", d_h=16, lr=1e-3, seed=0, max_epoch=5),
    out_root=f"{OUT_ROOT}_smoke",
    dataset_dir=DATASET_DIR,
    print_every=1,
)
smoke

## Full reproduction

One (table, dataset) block is 2 learning rates × 5 seeds = 10 runs of up to 1000 epochs. Small TU graphs leave a GPU mostly idle, so `JOBS = 2`–`4` concurrent runs is usually faster. REDDIT-BINARY and COLLAB are the expensive ones (REDDIT-B took ~2.3 h per run on an A100-class GPU).

Available datasets: `MUTAG`, `ENZYMES`, `PROTEINS`, `COLLAB`, `IMDB-BINARY`, `REDDIT-BINARY`.

In [ ]:
# 6. Run the grid (re-running skips finished runs)
DATASETS = ["MUTAG"]  # e.g. list(TU_DATASETS)
TABLES = (17, 18)  # 17 -> d_h=16, 18 -> d_h=4
GATE_ACT = "sigmoid"  # "gelu" for the Gated-GPS gate
JOBS = 2

results = run_sweep(
    DATASETS,
    tables=TABLES,
    gate_act=GATE_ACT,
    out_root=OUT_ROOT,
    dataset_dir=DATASET_DIR,
    wandb=USE_WANDB,
    wandb_entity=WANDB_ENTITY if USE_WANDB else None,
    jobs=JOBS,
    print_every=100,
)

In [ ]:
# 7. Table 17 / 18 summary (mean ± std over seeds, best LR per dataset)
print(summarize(OUT_ROOT))